In [ ]:
# @title ⚙️ Setup — run this first to load images
import sys

if 'google.colab' in sys.modules:
    from IPython.display import display, Javascript
    js = """
(function() {
  // Auto-detect GitHub repo from the Colab URL
  // e.g. colab.research.google.com/github/USER/REPO/blob/BRANCH/notebook.ipynb
  var m = window.location.href.match(
    /colab\.research\.google\.com\/github\/([^\/]+\/[^\/]+)\/blob\/([^\/]+)/
  );
  if (!m) { console.warn('Not opened from GitHub — images may not show.'); return; }
  var rawBase = 'https://raw.githubusercontent.com/' + m[1] + '/' + m[2];

  function patch() {
    document.querySelectorAll('img[src^="images/"]').forEach(function(img) {
      img.src = rawBase + '/' + img.getAttribute('src');
    });
  }
  patch();
  new MutationObserver(patch).observe(document.body, {childList:true, subtree:true});
  [300,800,1500,3000].forEach(function(d){ setTimeout(patch,d); });
  console.log('✅ Image patcher active →', rawBase);
})();
""";
    display(Javascript(js))
    print('✅ Images will load from your GitHub repo automatically.')
else:
    print('✅ Local mode — open with Jupyter Lab or Jupyter Notebook for images to display.')


# Part 1: Data Foundation & Preprocessing

---
## 1.1 Exploratory Data Analysis (EDA)

**Purpose:** Understand data structure, patterns, and quality issues

**Key Tasks:**
- Dataset overview
- Missing data analysis
- Outlier detection
- Statistical summaries
- Visualizations

### 🐼 Pandas Version


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('Datasets/data.csv')

# 1. Dataset Overview
print(df.shape)              # (rows, columns)
print(df.info())             # data types, non-null counts
print(df.head())             # first 5 rows
print(df.columns)            # column names

# 2. Missing Data Analysis
print(df.isnull().sum())     # count missing per column
print(df.isnull().mean() * 100)  # percentage missing

# Visualize missing data
import missingno as msno
msno.matrix(df)
plt.show()

# 3. Duplicate Detection
print(f"Duplicates: {df.duplicated().sum()}")

# 4. Statistical Summaries
print(df.describe())         # numerical columns stats
print(df.describe(include='object'))  # categorical stats

# 5. Outlier Detection (IQR method)
Q1 = df['column'].quantile(0.25)
Q3 = df['column'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['column'] < Q1 - 1.5*IQR) | (df['column'] > Q3 + 1.5*IQR)]
print(f"Outliers: {len(outliers)}")

# 6. Data Visualization
# Distribution plot
df['column'].hist(bins=30)
plt.show()

# Box plot for outliers
sns.boxplot(data=df, x='column')
plt.show()

# Correlation heatmap
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

# Pairplot
sns.pairplot(df)
plt.show()

# 7. Update values
df.loc[df['name'] == 'Bob', 'age'] = 31
df.loc[1, 'age'] = 31
df.iloc[1, 2] = 31

### ⚡ PySpark Version

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import when

spark = SparkSession.builder.appName("EDA").getOrCreate()

# Load data
df = spark.read.csv('Datasets/data.csv', header=True, inferSchema=True)

# 1. Dataset Overview
print((df.count(), len(df.columns)))
df.printSchema()
df.show()
print(df.columns)

# 2. Missing Data Analysis
df.select([sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()

df.select([
    (sum(col(c).isNull().cast('int')) * 100 / df.count()).alias(c)
    for c in df.columns
]).show()

# 3. Duplicate Detection
print("Duplicates:", df.count() - df.dropDuplicates().count())

# 4. Statistical Summaries
df.describe().show()
df.summary().show()

# 5. Outlier Detection (IQR Approximation)
quantiles = df.approxQuantile("column", [0.25, 0.75], 0)
Q1, Q3 = quantiles
IQR = Q3 - Q1

outliers = df.filter(
    (col("column") < Q1 - 1.5 * IQR) |
    (col("column") > Q3 + 1.5 * IQR)
)

print("Outliers:", outliers.count())

# 6. Update values
df = df.withColumn(
    "age",
    when(col("name") == "Bob", 31).otherwise(col("age"))
)

### 💡 Pro Insight (MLE Level)

In real pipelines:

* Use **PySpark → preprocessing at scale**
* Convert sample:

```python
pdf = df.sample(0.1).toPandas()
```

* Then use **Pandas + Seaborn for visualization**

---
## 1.2 Data Cleaning

**Purpose:** Remove/fix invalid, incomplete, or inconsistent data

### Handle Missing Data

#### 🐼 Pandas Version

In [ ]:
# Drop rows/columns with missing values
df.dropna()                   # drop rows with any NaN
df.dropna(axis=1)             # drop columns with any NaN
df.dropna(thresh=5)           # drop rows with <5 non-NaN values

# 1. Mean Imputation (for numerical, normally distributed)
df['column'].fillna(df['column'].mean(), inplace=True)

# 2. Median Imputation (for numerical, skewed data)
df['column'].fillna(df['column'].median(), inplace=True)

# 3. Mode Imputation (for categorical)
df['column'].fillna(df['column'].mode()[0], inplace=True)

# 4. Forward Fill / Backward Fill (for time series)
df['column'].fillna(method='ffill', inplace=True)
df['column'].fillna(method='bfill', inplace=True)

# 5. KNN Imputation
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# 6. MICE (Multiple Imputation by Chained Equations)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imputer = IterativeImputer(random_state=0)
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# 7. SimpleImputer (sklearn)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')  # 'mean', 'median', 'most_frequent', 'constant'
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

#### ⚡ PySpark Version

In [ ]:
from pyspark.sql.functions import col, sum, mean
from pyspark.sql.functions import when
from pyspark.sql import functions as F

# Drop rows/columns with missing values
df.na.drop().show()
df.na.drop(subset=["age", "salary"]).show()
df.na.drop(thresh=5).show()

# Drop columns with no nulls (your logic preserved)
null_count = df.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
]).collect()[0].asDict()

df.select([c for c in df.columns if null_count[c] == 0]).show()

# 1. Mean Imputation
mean_fill = df.select([
    F.round(mean(col(c)), 3).alias(c)
    for c in ['Age', 'Salary']
]).collect()[0].asDict()

df.fillna(mean_fill).show()

# 2. Median Imputation
median_fill = df.approxQuantile(['Age', 'Salary'], [0.5], 0)

# OR

median_fill = df.select([
    median(col(c)).cast('int').alias(c)
    for c in ['Age','Salary']
]).collect()[0].asDict()

df.fillna(median_fill).show()

# 3. Mode Imputation
mode_fill = {
    c: df.groupBy(c).count().orderBy('count', ascending=False).first()[0]
    for c in df.columns
}

df.fillna(mode_fill).show()

### Remove Duplicates

#### 🐼 Pandas Version

In [ ]:
df = pd.read_csv('Datasets/user.csv')

# Remove exact duplicates
df_clean = df.drop_duplicates()

# Remove duplicates based on specific columns
df_clean = df.drop_duplicates(subset=['col1', 'col2'], keep='first')


#### ⚡ PySpark Version

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv('Datasets/user.csv', header=True, inferSchema=True)

# Remove exact duplicates
df_clean = df.dropDuplicates()

# Remove duplicates based on specific columns
df_clean = df.dropDuplicates(["email"])

### Handle Outliers

#### **Method 1: IQR**


##### 🧠 What is IQR?

IQR = **Q3 − Q1**

* **Q1 (25th percentile)** → lower quartile
* **Q3 (75th percentile)** → upper quartile
* IQR measures the **spread of the middle 50% data**


#### Outlier Rule

Any value is an outlier if:

* **Lower bound** = Q1 − 1.5 × IQR
* **Upper bound** = Q3 + 1.5 × IQR

Values outside this range = **outliers**



##### 📊 Example Dataset

```
data = [10, 12, 14, 15, 18, 19, 21, 22, 100]
```

Clearly, `100` looks suspicious — let’s verify mathematically.


##### ✅ Step 1: Sort Data

Already sorted:

```
[10, 12, 14, 15, 18, 19, 21, 22, 100]
```

##### ✅ Step 2: Find Median (Q2)

Total elements = 9
Median = 5th value → **18**

##### ✅ Step 3: Split Data

* Lower half → `[10, 12, 14, 15]`
* Upper half → `[19, 21, 22, 100]`


##### ✅ Step 4: Find Q1 and Q3

* Q1 = median of `[10, 12, 14, 15]`
  → (12 + 14) / 2 = **13**

* Q3 = median of `[19, 21, 22, 100]`
  → (21 + 22) / 2 = **21.5**


##### ✅ Step 5: Calculate IQR

IQR = Q3 − Q1 =

```
21.5 − 13 = 8.5
```

##### ✅ Step 6: Compute Bounds

* Lower bound = 13 − (1.5 × 8.5) = **0.25**
* Upper bound = 21.5 + (1.5 × 8.5) = **34.25**


##### 🚨 Step 7: Detect Outliers

Check values:

* Any value < 0.25 → ❌ None
* Any value > 34.25 → ✅ **100 is an outlier**


##### 🎯 Final Result

```
Outliers = [100]
```

##### 💻 Python Code

In [ ]:
import numpy as np

data = np.array([10, 12, 14, 15, 18, 19, 21, 22, 100])

Q1 = np.percentile(data, 25) # df['column'].quantile(0.25)
Q3 = np.percentile(data, 75) # df['column'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data < lower_bound) | (data > upper_bound)]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Outliers:", outliers)

#### **Method 2: Z-score**

##### 🧠 What is Z-score?
Z-score tells **how many standard deviations a value is away from the mean**

<div align="center">
  <img src="images/Part1_Data_Foundation_Preprocessing_cell25_img0_ebb18f1e.png" alt="Z Score" style="max-width:90%;height:auto;" />
</div>

Where:

* **x** = data point
* **μ (mu)** = mean of the dataset
* **σ (sigma)** = standard deviation

##### 🚨 Outlier Rule (Z-score)

* If **|Z| > 3** → Outlier (most common threshold)
* Sometimes **2.5** is also used


##### 📊 Example Dataset

```python
data = [10, 12, 14, 15, 18, 19, 21, 22, 100]
```

##### ✅ Step-by-Step Calculation

##### Step 1: Mean (μ)

```python
mean = (10 + 12 + 14 + 15 + 18 + 19 + 21 + 22 + 100) / 9
     = 231 / 9
     = 25.67 (approx)
```

---

##### Step 2: Standard Deviation (σ)

You don’t need to manually compute in interviews, but roughly:

```python
std ≈ 26.7
```

👉 Notice: **100 is pulling mean & std upward**

---

##### Step 3: Compute Z-score for 100

```python
z = (100 - 25.67) / 26.7 ≈ 2.78
```

---

##### 🚨 Step 4: Check Outlier

* |2.78| < 3 → ❌ **Not detected as outlier**

---

##### 🎯 Result (Z-score)

```python
Outliers = []
```

👉 Even though 100 looks extreme, **Z-score FAILED to detect it**

---

##### 💥 Why did Z-score fail?

Because:

* Mean got shifted ↑ (25.67 instead of ~18)
* Std deviation got inflated ↑
* So Z-score becomes smaller

👉 **Z-score is sensitive to extreme values**



##### 💻 Python Code

In [ ]:
import numpy as np

data = np.array([10, 12, 14, 15, 18, 19, 21, 22, 100])

mean = np.mean(data)
std = np.std(data)

z_scores = np.abs((data - mean) / std) # np.abs(stats.zscore(df['column']))

outliers = data[z_scores > 3]

print("Mean:", mean)
print("Std:", std)
print("Z-scores:", z_scores)
print("Outliers:", outliers)

#### ⚔️ IQR vs Z-score (Important)

| Feature                     | IQR Method      | Z-score Method |
| --------------------------- | --------------- | -------------- |
| Based on                    | Median (Q1, Q3) | Mean & Std     |
| Robust to outliers          | ✅ Yes           | ❌ No           |
| Works for skewed data       | ✅ Yes           | ❌ No           |
| Assumes normal distribution | ❌ No            | ✅ Yes          |
| Our example result          | 100 detected ✅  | Not detected ❌ |

---

###### 🧠 Intuition Difference

##### IQR

* Looks at **middle 50% of data**
* Ignores extremes → more stable

##### Z-score

* Uses **entire dataset**
* Gets affected by extreme values
        
##### When to use what?

* Use **IQR** →

  * Skewed data
  * Real-world messy datasets
  * Robust detection

* Use **Z-score** →

  * Normally distributed data
  * Statistical modeling

#### Method 3: Winsorization (Cap outliers)

##### 🧠 What is Winsorization?

👉 A technique where:

* Extreme low values → replaced with a lower threshold
* Extreme high values → replaced with an upper threshold

✅ **Outliers are not removed, just “pulled back”**

##### 📊 Example Dataset

```python
[10, 12, 14, 15, 18, 19, 21, 22, 100]
```

Assume:

* 1% ≈ 10
* 99% ≈ 22

##### After Winsorization

```python
[10, 12, 14, 15, 18, 19, 21, 22, 22]
```

👉 **100 → becomes 22**

---

##### ✅ Step 1: Find thresholds

```python
lower = df['column'].quantile(0.01)  # 1st percentile
upper = df['column'].quantile(0.99)  # 99th percentile
```

* `0.01` → bottom 1% value
* `0.99` → top 1% value

👉 You’re defining **acceptable range**

---

##### ✅ Step 2: Clip values

```python
df['column'] = df['column'].clip(lower, upper)
```

This does:

* If value < lower → set it = lower
* If value > upper → set it = upper
* Otherwise → keep as is

##### 💻 Python Code

In [ ]:
lower = df['column'].quantile(0.01)
upper = df['column'].quantile(0.99)
df['column'] = df['column'].clip(lower, upper)

##### ⚡ PySpark Code

In [ ]:
from pyspark.sql.functions import col, when

lower = df.approxQuantile("column", [0.01], 0)[0]
upper = df.approxQuantile("column", [0.99], 0)[0]

df = df.withColumn(
    "column",
    when(col("column") < lower, lower)
    .when(col("column") > upper, upper)
    .otherwise(col("column"))
)

##### 🔥 Why use this?

##### Instead of:

* ❌ Dropping rows → lose data
* ❌ Ignoring outliers → skew model

##### We do:

* ✅ Keep all rows
* ✅ Reduce impact of extreme values

##### 💡 When to Use Winsorization

Use it when:

* You **don’t want to lose data**
* Dataset is small or important (finance, logs)
* Preparing data for ML models (especially linear models)

#### ⚔️ Winsorization vs IQR vs Z-score

| Method        | Action                              |
| ------------- | ----------------------------------- |
| IQR           | Detect & optionally remove outliers |
| Z-score       | Detect based on standard deviation  |
| Winsorization | **Modify values (cap them)**        |

#### 🚀 Interview Insight (Very Important)

👉 If interviewer asks:

**“What do you do after detecting outliers?”**

Answer:

* Remove (if noise)
* Transform (log scaling)
* **Cap (Winsorization)** ✅ (best practical answer)

## 1.3 Feature Engineering

**Purpose:** Create new features to improve model performance

### Creating New Features

#### 🐼 Pandas Version

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

# 1. Interaction Features
df['feature1_x_feature2'] = df['feature1'] * df['feature2']
df['feature1_div_feature2'] = df['feature1'] / (df['feature2'] + 1e-10)
df['feature_sum'] = df['feature1'] + df['feature2']

# 2. Polynomial Features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df[['feature1', 'feature2']])
# Output: feature1, feature2, feature1^2, feature1*feature2, feature2^2

# 3. Binning / Discretization

# Equal-width binning
df['age_binned'] = pd.cut(
    df['age'],
    bins=5,
    labels=['very_young', 'young', 'middle', 'old', 'very_old']
)

# Custom bins
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 100],
    labels=['child', 'young', 'adult', 'senior']
)

# Quantile-based binning
df['income_quartile'] = pd.qcut(
    df['income'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

# 4. Aggregations (group statistics)
df['mean_by_category'] = df.groupby('category')['value'].transform('mean')
df['sum_by_category'] = df.groupby('category')['value'].transform('sum')
df['count_by_category'] = df.groupby('category')['value'].transform('count')

# 5. Time-based Features
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter
df['hour'] = df['date'].dt.hour

df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

# Time difference
df['days_since'] = (pd.Timestamp.now() - df['date']).dt.days

# 6. Mathematical Transformations
df['log_feature'] = np.log1p(df['feature'])
df['sqrt_feature'] = np.sqrt(df['feature'])
df['square_feature'] = df['feature'] ** 2
df['exp_feature'] = np.exp(df['feature'])

#### ⚡ PySpark Version

In [ ]:
from pyspark.sql.functions import col, mean, sum as _sum, count, when, dayofweek
from pyspark.sql.window import Window

# 1. Interaction Features
df = df.withColumn('feature1_x_feature2', col('feature1') * col('feature2'))
df = df.withColumn('feature1_div_feature2', col('feature1') / (col('feature2') + 1e-10))
df = df.withColumn('feature_sum', col('feature1') + col('feature2'))

# 2. Aggregations using Window
window_spec = Window.partitionBy('category')

df = df.withColumn('mean_by_category', mean('value').over(window_spec))
df = df.withColumn('sum_by_category', _sum('value').over(window_spec))
df = df.withColumn('count_by_category', count('value').over(window_spec))

# 3. GroupBy (for reference)
df.groupBy('category').agg(mean('value')).show()

# 4. Time-based Features
df = df.withColumn('dayofweek', dayofweek(col('date').cast('timestamp')))

df = df.withColumn(
    'is_weekend',
    when(col('dayofweek').isin([1, 7]), 1).otherwise(0)  # Spark: Sunday=1, Saturday=7
)

# 5. Mathematical Transformations
from pyspark.sql.functions import log1p, sqrt, exp

df = df.withColumn('log_feature', log1p(col('feature')))
df = df.withColumn('sqrt_feature', sqrt(col('feature')))
df = df.withColumn('square_feature', col('feature') ** 2)
df = df.withColumn('exp_feature', exp(col('feature')))

### Feature Selection Methods

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# 1. Filter Methods (Statistical tests)
# Chi-square test (for categorical target)
selector = SelectKBest(chi2, k=10)
X_selected = selector.fit_transform(X, y)

# ANOVA F-test
selector = SelectKBest(f_classif, k=10)
X_selected = selector.fit_transform(X, y)

# Mutual Information
selector = SelectKBest(mutual_info_classif, k=10)
X_selected = selector.fit_transform(X, y)

# 2. Wrapper Methods
# Recursive Feature Elimination (RFE)
model = RandomForestClassifier()
rfe = RFE(estimator=model, n_features_to_select=10)
X_selected = rfe.fit_transform(X, y)
print("Selected features:", X.columns[rfe.support_])

# 3. Embedded Methods
# L1-based feature selection (Lasso)
from sklearn.linear_model import LassoCV
model = LassoCV(cv=5)
selector = SelectFromModel(model)
X_selected = selector.fit_transform(X, y)

# Tree-based feature importance
model = RandomForestClassifier()
model.fit(X, y)
importances = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
importances = importances.sort_values('importance', ascending=False)
print(importances)

# 4. Correlation-based removal (remove highly correlated features)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
X_reduced = X.drop(columns=to_drop)

# 5. Variance Threshold (remove low variance features)
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)

---
## 1.4 Encoding Categorical Data

**Purpose:** Convert categorical variables to numerical format

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import category_encoders as ce

# 1. Label Encoding (ordinal or binary categorical)
# Use when: categories have order (low, medium, high) or for binary (yes/no)
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])
# Decode: le.inverse_transform(df['category_encoded'])

# Manual ordinal encoding
df['size'] = df['size'].map({'small': 1, 'medium': 2, 'large': 3})

# 2. One-Hot Encoding (nominal categorical with few categories)
# Use when: no order, creates n columns for n categories
df_encoded = pd.get_dummies(df, columns=['category'], drop_first=True)

# sklearn method
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False)
encoded = ohe.fit_transform(df[['category']])

# 3. Target Encoding (mean encoding)
# Use when: high cardinality, supervised learning
# Replaces category with mean of target for that category
encoder = ce.TargetEncoder(cols=['category'])
df['category_encoded'] = encoder.fit_transform(df['category'], df['target'])

# Manual target encoding with smoothing
target_mean = df.groupby('category')['target'].mean()
df['category_encoded'] = df['category'].map(target_mean)

# 4. Binary Encoding (for high cardinality)
# Converts to binary digits, uses fewer columns than one-hot
encoder = ce.BinaryEncoder(cols=['category'])
df_encoded = encoder.fit_transform(df)

# 5. Hash Encoding (for very high cardinality)
# Fixed number of features, risk of collision
encoder = ce.HashingEncoder(cols=['category'], n_components=8)
df_encoded = encoder.fit_transform(df)

# 6. Frequency Encoding
# Replace with frequency of occurrence
freq_map = df['category'].value_counts(normalize=True).to_dict()
df['category_freq'] = df['category'].map(freq_map)

# sklearn method
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder()
df['category_encoded'] = oe.fit_transform(df[['category']])

**Encoding Comparison:**

| Method | Use Case | Pros | Cons |
|--------|----------|------|------|
| Label | Ordinal/Binary | Simple, no new columns | Implies order |
| One-Hot | Low cardinality | No false relationships | High dimensionality |
| Target | High cardinality | Captures target relation | Risk of overfitting |
| Binary | Medium-high cardinality | Fewer columns than one-hot | Less interpretable |
| Hash | Very high cardinality | Fixed size | Collision risk |
| Frequency | Any cardinality | Simple, captures popularity | Loses category identity |

---
## 1.5 Feature Scaling

**Purpose:** Normalize feature ranges for algorithms sensitive to scale

**When to use:** KNN, SVM, Neural Networks, PCA, Gradient Descent-based algorithms

**When NOT to use:** Tree-based models (Decision Trees, Random Forest, XGBoost)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer

# 1. Standardization (Z-score normalization)
# Formula: (x - mean) / std
# Result: mean=0, std=1
# Use when: data is normally distributed, for algorithms assuming normal distribution
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Inverse: scaler.inverse_transform(X_scaled)

# 2. Min-Max Normalization (Rescaling)
# Formula: (x - min) / (max - min)
# Result: values in [0, 1] or custom range
# Use when: need bounded values, neural networks
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X)

# Custom range
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler.fit_transform(X)

# 3. Robust Scaling
# Formula: (x - median) / IQR
# Uses median and IQR (interquartile range)
# Use when: data has outliers (robust to outliers)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# 4. Unit Vector Scaling (Normalization)
# Formula: x / ||x|| (where ||x|| is L2 norm)
# Result: each sample has unit norm
# Use when: direction matters more than magnitude, text/NLP
scaler = Normalizer(norm='l2')  # 'l1', 'l2', or 'max'
X_scaled = scaler.fit_transform(X)

# Manual L2 normalization
X_scaled = X / np.linalg.norm(X, axis=1, keepdims=True)

**Scaling Comparison:**

| Method | Range | Outlier Sensitive | Use Case |
|--------|-------|-------------------|----------|
| StandardScaler | Unbounded | Yes | Normal distribution, PCA |
| MinMaxScaler | [0,1] or custom | Yes | Neural networks, bounded values |
| RobustScaler | Unbounded | No | Data with outliers |
| Normalizer | Unit norm | N/A | Text data, cosine similarity |

**Important Notes:**
- Always fit on training data only, then transform both train and test
- Scale after train-test split to avoid data leakage

In [ ]:
from sklearn.model_selection import train_test_split

# Correct approach
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled = scaler.transform(X_test)        # transform test using train stats

---
## 1.6 Model Prediction Methods

**Purpose:** Understand different prediction methods for various use cases

### predict() vs predict_proba()

**predict():**
- Returns the predicted class label
- Output: discrete class (0, 1, or class names)
- Use when: you need final decision/classification

**predict_proba():**
- Returns probability for each class
- Output: array of probabilities [P(class0), P(class1), ...]
- Use when: you need confidence scores, custom thresholds, or ranking

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Train a classifier
model = LogisticRegression()
model.fit(X_train, y_train)

# 1. predict() - Get class labels
y_pred = model.predict(X_test)
print(y_pred)  # Output: [0, 1, 1, 0, ...]

# 2. predict_proba() - Get probabilities
y_proba = model.predict_proba(X_test)
print(y_proba)  # Output: [[0.8, 0.2], [0.3, 0.7], ...]
#                         [P(class0), P(class1)]

# 3. predict_log_proba() - Get log probabilities
y_log_proba = model.predict_log_proba(X_test)
# Useful for numerical stability

# 4. decision_function() - Get decision scores (not all models)
# Available for: SVM, Logistic Regression
y_scores = model.decision_function(X_test)
# Distance from decision boundary

### Common Use Cases

In [ ]:
# Use Case 1: Standard classification
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

# Use Case 2: Custom threshold (e.g., for imbalanced data)
y_proba = model.predict_proba(X_test)
threshold = 0.3
y_pred_custom = (y_proba[:, 1] >= threshold).astype(int)

# Use Case 3: ROC curve and AUC (needs probabilities)
from sklearn.metrics import roc_curve, roc_auc_score
y_proba = model.predict_proba(X_test)[:, 1]  # probability of positive class
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

# Use Case 4: Confidence-based filtering
y_proba = model.predict_proba(X_test)
confidence = np.max(y_proba, axis=1)  # max probability
high_confidence = confidence > 0.9
y_pred_confident = model.predict(X_test[high_confidence])

# Use Case 5: Multi-class probabilities
y_proba = model.predict_proba(X_test)
# For 3 classes: [[0.7, 0.2, 0.1], [0.1, 0.1, 0.8], ...]
predicted_class = np.argmax(y_proba, axis=1)  # class with highest prob

# Use Case 6: Top-N predictions
y_proba = model.predict_proba(X_test)
top_3_classes = np.argsort(y_proba, axis=1)[:, -3:]  # top 3 classes

**Method Availability by Model:**

| Model | predict() | predict_proba() | decision_function() |
|-------|-----------|-----------------|---------------------|
| Logistic Regression | ✓ | ✓ | ✓ |
| SVM | ✓ | ✓ (if probability=True) | ✓ |
| Decision Tree | ✓ | ✓ | ✗ |
| Random Forest | ✓ | ✓ | ✗ |
| KNN | ✓ | ✓ | ✗ |
| Naive Bayes | ✓ | ✓ | ✗ |

---
## Complete Preprocessing Pipeline Example

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Separate features by type
numerical_features = ['age', 'income', 'score']
categorical_features = ['category', 'city', 'type']

# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full pipeline with model
from sklearn.ensemble import RandomForestClassifier

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
full_pipeline.fit(X_train, y_train)

# Predict
y_pred = full_pipeline.predict(X_test)
print(f"Accuracy: {full_pipeline.score(X_test, y_test):.3f}")

## Quick Reference Checklist

**Before Training Any Model:**

1. ✓ EDA: Understand data shape, types, distributions
2. ✓ Handle missing values: Choose appropriate imputation
3. ✓ Handle outliers: Remove, cap, or transform
4. ✓ Encode categorical: Choose encoding based on cardinality
5. ✓ Feature engineering: Create domain-specific features
6. ✓ Feature selection: Remove redundant/irrelevant features
7. ✓ Feature scaling: Scale if using distance-based algorithms
8. ✓ Train-test split: Always split before scaling
9. ✓ Use pipelines: Ensure reproducibility and prevent leakage

**Common Mistakes to Avoid:**
- Scaling before train-test split (data leakage)
- Using one-hot encoding for high cardinality features
- Not handling outliers before scaling
- Removing duplicates after splitting
- Using mean imputation for skewed data